# CSC311 Final Project: Proposal - Data Exploration

## Description
This notebook contains the initial exploratory data analysis (EDA) and preprocessing pipeline for the CSC311 Fall 2026 Machine Learning Project. Our final goal is to build a 3-class classifier that predicts a sport (Soccer, Baseball, or Basketball) based on a dataset of student survey responses.

To prepare our data for model training and satisfy the project requirements, we followed these steps:

1. **Feature Analysis:** Examining feature types (numeric, Likert-scale, multi-select, free text), their distributions, and the overall class balance.
2. **Data Cleaning:** Identifying missing values, outliers, and inconsistencies, and implementing strategies to handle them.
3. **Preprocessing Strategy:** Designing and justifying the transformations required for model compatibility, including normalization, encoding, and text representation.
4. **Data Splitting:** Explaining how the test set was set aside and never used during
exploration.

### Imports, Data Loading, and Train/Test Split

In [6]:
import re
from collections import Counter
import numpy as np
import pandas as pd
# from sklearn.model_selection import train_test_split

# 1. Load the dataset
df = pd.read_csv('csc311_project_training_202609.csv')

# 2. Rename columns into short, descriptive names
COLUMN_NAMES = {
    df.columns[1]: 'ease_rating',
    df.columns[2]: 'strategy_importance',
    df.columns[3]: 'high_injury_risk',
    df.columns[4]: 'individual_brilliance_outweigh_teamwork',
    df.columns[5]: 'locations_to_play',
    df.columns[6]: 'seasons_to_play',
    df.columns[7]: 'music_to_play',
    df.columns[8]: 'food_drink',
    df.columns[9]: 'color_associated_with_sport',
    df.columns[10]: 'song_associated_with_sport',
    df.columns[11]: 'feeling_when_watching_sport',
    df.columns[12]: 'ticket_price'
}
df = df.rename(columns=COLUMN_NAMES)

# 3. Drop rows where ALL feature columns are missing (NULL)
df_cleaned = df.dropna(subset=df.columns[1:13], how='all')
print(f"Dropped {len(df) - len(df_cleaned)} completely empty rows.")

# 4. Split BEFORE Exploration/Preprocessing to prevent data leakage
# train_df, test_df = train_test_split(df_cleaned, test_size=0.2, random_state=42)
train_df = df
# print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

Dropped 7 completely empty rows.


### Raw Text Exploration (Training Set Only)

In [11]:
def get_top_phrases(column, top_n=20):
    """Cleans a text column and returns the most common comma-separated phrases."""
    cleaned = column.fillna("").str.lower().str.replace(r"[^\w\s,]", "", regex=True).str.split(",")
    flat_list = [item.strip() for sublist in cleaned for item in sublist if item.strip()]
    return Counter(flat_list).most_common(top_n)

# Explore ONLY on train_df!
print("Top Feelings:", get_top_phrases(train_df["feeling_when_watching_sport"], 20), "\n")
print("Top Foods:", get_top_phrases(train_df["food_drink"], 20), "\n")
print("Top Colors:", get_top_phrases(train_df["color_associated_with_sport"], 20), "\n")
print("Top Songs:", get_top_phrases(train_df["song_associated_with_sport"], 20), "\n")


Top Feelings: [('excited', 207), ('bored', 89), ('neutral', 29), ('happy', 22), ('energetic', 16), ('confused', 16), ('nervous', 14), ('exciting', 14), ('anxious', 13), ('excitement', 12), ('relaxed', 11), ('boring', 10), ('entertained', 10), ('energized', 10), ('interested', 10), ('tense', 9), ('nothing', 8), ('good', 8), ('stressed', 7), ('thrilled', 7)] 

Top Foods: [('beer', 182), ('popcorn', 95), ('hot dogs', 93), ('pizza', 76), ('nachos', 65), ('soda', 64), ('fries', 64), ('chips', 57), ('coke', 56), ('hotdogs', 41), ('hot dog', 40), ('burgers', 38), ('chicken wings', 37), ('hotdog', 34), ('peanuts', 29), ('wings', 24), ('fried chicken', 21), ('water', 18), ('soft drinks', 18), ('gatorade', 16)] 

Top Colors: [('blue', 191), ('red', 151), ('orange', 148), ('green', 144), ('white', 107), ('brown', 53), ('black', 38), ('yellow', 30), ('black and white', 10), ('purple', 7), ('white and black', 6), ('gold', 5), ('blue and red', 5), ('green and white', 3), ('red and blue', 3), ('beige

### Encoding Mappings & Dictionaries
Mappings for Likert scales (Strongly Disagree to Strongly Agree) into numeric values (1 to 5) and keyword dictionaries derived from exploring the top 100 words for binary text classification.

In [13]:
# This is a list of the columns containing Likert scale responses
LIKERT_COLS = [
    "strategy_importance",
    "high_injury_risk",
    "individual_brilliance_outweigh_teamwork",
]

LIKERT_MAP = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Neutral": 3,
    "Agree": 4,
    "Strongly agree": 5,
}

# This map contains the responses to the multiple-choice questions
MULTI_CHOICE_MAP = {
    "seasons_to_play": ["Winter", "Spring", "Summer", "Fall"],
    "locations_to_play": [
        "Indoor facility", "Outdoor public space", "School or university facility",
        "Community recreation facility", "Private sports facility", "At home or nearby"
    ],
    "music_to_play": [
        "Pop", "Rock", "Hip-hop or rap", "Electronic or dance", "Latin",
        "Country", "Classical or jazz", "Folk or traditional", "Other", "No particular type"
    ]
}

FOOD_CATEGORIES = {
    "food_beer": ["beer", "beers", "alcohol"],
    "food_hotdog": ["hot dog", "hotdog", "hot dogs", "hotdogs", "sausage"],
    "food_pizza": ["pizza"],
    "food_popcorn": ["popcorn", "pop corn", "popcorns"],
    "food_nachos": ["nachos", "loaded nachos"],
    "food_soda_cola": ["soda", "coke", "pop", "cola", "cocacola", "coca cola", "soft drinks", "softdrink", "sprite", "pepsi", "lemonade"],
    "food_fries": ["fries", "french fries"],
    "food_chips": ["chips", "potato chips"],
    "food_burger": ["burger", "burgers", "hamburger", "hamburgers", "sliders"],
    "food_wings": ["wings", "chicken wings"],
    "food_chicken": ["chicken", "fried chicken", "chicken tenders"],
    "food_peanuts_nuts": ["peanuts", "nuts", "sunflower seeds"],
    "food_water": ["water"],
    "food_sports_drinks": ["gatorade"],
    "food_poutine": ["poutine"],
    "food_sandwiches": ["sandwiches", "sandwich", "shawarma"],
    "food_pretzels": ["pretzels"],
    "food_tacos": ["tacos"]
}

COLOR_CATEGORIES = {
    "color_green": ["green"],
    "color_blue": ["blue", "navy"],
    "color_orange": ["orange"],
    "color_red": ["red", "burgundy", "maroon"],
    "color_white": ["white"],
    "color_black": ["black"],
    "color_brown": ["brown", "wood", "wooden", "wood-toned"],
    "color_yellow_gold": ["yellow", "gold", "golden"],
    "color_purple": ["purple"]
}

SONG_CATEGORIES = {
    "song_waka_waka": ["waka waka", "shakira"],
    "song_ball_game": ["take me out to the ball game", "ball game"],
    "song_we_will_rock_you": ["we will rock you", "queen"],
    "song_see_you_again": ["see you again"],
    "song_sirius": ["sirius", "alan parsons"],
    "song_champions": ["we are the champions", "champion"],
    "song_space_jam": ["space jam", "quad city"],
    "song_drake_hiphop": ["drake", "sicko mode", "travis scott", "kendrick", "god's plan", "forever"],
    "song_wavin_flag": ["wavin' flag", "wavin flag", "k'naan"],
}

SENTIMENT_CATEGORIES = {
    "feeling_excited": ["excited", "thrilled", "hyped", "ecstatic", "passionate", "energetic", "electric", "energized", "immersive", "immersed"],
    "feeling_fast_paced": ["fast-paced", "fast pace", "fastpaced", "quick pace", "fast"],
    "feeling_bored": ["bored", "boring", "indifferent", "uninterested", "slow", "monotonous"],
    "feeling_tense_anxious": ["tense", "nervous", "anxious", "on edge", "on-edge", "stressed", "restless"],
    "feeling_relaxed_calm": ["relaxed", "calm", "peaceful", "comfortable"],
    "feeling_confused": ["confused", "don't know", "dont know", "doesn't understand", "do not understand", "jargon"],
    "feeling_happy": ["happy", "joy", "joyful", "good", "glad", "pleased"],
    "feeling_neutral": ["neutral", "fine", "okay", "ok"]
}

### Preprocessing

In [18]:
def extract_price(text):
    if pd.isna(text):
        return 0.0
    text_lower = str(text).lower()
    numbers = re.findall(r'\d[\d,]*\.?\d*', text_lower.replace(',', ''))
    if not numbers:
        return 0.0
    float_nums = [float(n) for n in numbers]
    if len(float_nums) >= 2:
        return np.mean(float_nums[:2])
    return float_nums[0]

def preprocess_dataset(data):
    """Transform raw survey dataframe into a clean numeric feature matrix."""
    out = pd.DataFrame(index=data.index)

    # 1. Target column
    target_col = data.columns[13]
    out[target_col] = data[target_col]

    # 2. Numeric & Likert features
    out["ease_rating"] = pd.to_numeric(data["ease_rating"], errors="coerce")
    for col in LIKERT_COLS:
        out[f"likert_{col}"] = data[col].map(LIKERT_MAP)

    # 3. Multi-choice comma-separated features (Seasons, Locations, Music)
    for col_name, options in MULTI_CHOICE_MAP.items():
        split_series = data[col_name].fillna("").str.split(",")
        for option in options:
            out[option] = split_series.apply(lambda s: int(option in [item.strip() for item in s]))

    # 4. Keyword category features (Food, Color, Song, Sentiment)
    keyword_tasks = [
        ("food_drink", FOOD_CATEGORIES),
        ("color_associated_with_sport", COLOR_CATEGORIES),
        ("song_associated_with_sport", SONG_CATEGORIES),
        ("feeling_when_watching_sport", SENTIMENT_CATEGORIES),
    ]
    for col_name, category_dict in keyword_tasks:
        text_series = data[col_name].fillna("").str.lower()
        for category, keywords in category_dict.items():
            out[category] = text_series.apply(lambda text: int(any(kw in text for kw in keywords)))

    # 5. Ticket Price extraction
    out["ticket_price"] = data["ticket_price"].apply(extract_price)

    # return out
    print(out.head())
preprocess_dataset(df)

        label  ease_rating  likert_strategy_importance  \
0      Soccer          4.0                         4.0   
1    Baseball          6.0                         3.0   
2  Basketball          7.0                         5.0   
3      Soccer          4.0                         4.0   
4    Baseball          8.0                         3.0   

   likert_high_injury_risk  likert_individual_brilliance_outweigh_teamwork  \
0                      4.0                                             1.0   
1                      2.0                                             4.0   
2                      3.0                                             5.0   
3                      4.0                                             5.0   
4                      4.0                                             5.0   

   Winter  Spring  Summer  Fall  Indoor facility  ...  song_wavin_flag  \
0       1       1       1     1                1  ...                0   
1       1       1       1     1   